# __Inteligencia Artificial - IUA 2026__

### *Pipeline EDA*

### Integrantes
* **Celeste Antonella Badami** — cbadami845@alumnos.iua.edu.ar
* **Benjamin Vargas** — bvargas161@alumnos.iua.edu.ar

---


## 1. Introducción
Este documento implementa un pipeline EDA para llevar las muestras a un dataset util para entrenamiento.

**Etapas:** 

2. Configuración y carga
3. Visión general
4. Calidad de datos
5. Limpieza y tipos
6. Variables temporales
7. Valores faltantes
8. Desvíos de la traza
9. Ingeniería de variables
10. Exportación del dataset procesado


## 2. Configuración y carga
Se definen las rutas y parámetros en un solo lugar para poder modificarlos fácilmente. El dataset se descarga del Drive y se coloca en `data/`.

In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# --- Parámetros del pipeline (modificables) ---
DATA_PATH = Path("../data/sample.csv")
OUTPUT_PATH = Path("../data/processed/dataset_procesado.csv")
ELIMINAR_DESVIOS = True    # True = descarta filas con es_desvio == True
UMBRAL_CORTE_S = 600       # si pasan más segundos entre muestras de un bus, se corta el tramo
VEL_MAX_KMH = 80           # velocidad estimada máxima creíble; por encima es ruido

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()   # trabajamos sobre una copia, el crudo queda intacto
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

Filas: 2132 | Columnas: 29


,id,bearing,current_status,direction_id,occupancy_status,latitude,longitude,bus_id,route_id,route_pattern_id,shape_id,agency_id,line_id,distancia_en_traza,distancia_fuera_traza,es_desvio,date_time,timezone,temperature_2m,precipitation,relative_humidity_2m,weather_code,is_day,current_speed,free_flow_speed,current_travel_time,freeflow_travel_time,confidence,road_closure
0,1,315.0,IN_TRANSIT_TO,0,FULL,42.359920,-71.094600,y1923,1,1-_-0,10154,1,line-1,4.572264,4.154841,False,2026-11-04 15:08:21.320843+00,America/New_York,19.606501,0.0,65.0,2.0,True,18,18,216,216,1.0,False
1,2,113.0,IN_TRANSIT_TO,1,MANY_SEATS_AVAILABLE,42.371286,-71.116519,y1918,1,1-_-1,10156,1,line-1,0.503092,7.303025,False,2026-11-04 15:08:28.354383+00,America/New_York,19.769501,0.0,63.0,2.0,True,18,18,125,125,1.0,False
2,3,43.0,IN_TRANSIT_TO,1,MANY_SEATS_AVAILABLE,42.349551,-71.088831,y1878,1,1-_-1,10156,1,line-1,3.984350,1.779268,False,2026-11-04 15:08:36.144629+00,America/New_York,19.763000,0.0,63.0,2.0,True,27,33,129,105,1.0,False
3,4,135.0,IN_TRANSIT_TO,1,MANY_SEATS_AVAILABLE,42.342210,-71.084290,y1831,1,1-_-1,10156,1,line-1,4.818362,8.385059,False,2026-11-04 15:08:43.847655+00,America/New_York,19.525999,0.0,65.0,2.0,True,12,15,105,84,1.0,False
4,5,145.0,STOPPED_AT,1,MANY_SEATS_AVAILABLE,42.339168,-71.080314,y1830,1,1-_-1,10156,1,line-1,5.295785,0.647673,False,2026-11-04 15:08:52.250099+00,America/New_York,19.500000,0.0,65.0,2.0,True,17,19,252,225,1.0,False


## 3. Visión general
Revisa tipos de dato y estadísticas descriptivas para entender qué contiene cada columna.

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     2132 non-null   int64  
 1   bearing                1653 non-null   float64
 2   current_status         2132 non-null   str    
 3   direction_id           2132 non-null   int64  
 4   occupancy_status       2131 non-null   str    
 5   latitude               2132 non-null   float64
 6   longitude              2132 non-null   float64
 7   bus_id                 2132 non-null   str    
 8   route_id               2132 non-null   int64  
 9   route_pattern_id       2132 non-null   str    
 10  shape_id               2132 non-null   int64  
 11  agency_id              2132 non-null   int64  
 12  line_id                2132 non-null   str    
 13  distancia_en_traza     2132 non-null   float64
 14  distancia_fuera_traza  2132 non-null   float64
 15  es_desvio      

In [22]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,2132.0,NaN,NaN,NaN,1066.5,615.599708,1.0,533.75,1066.5,1599.25,2132.0
bearing,1653.0,NaN,NaN,NaN,198.119177,99.204113,0.0,123.0,180.0,307.0,359.0
current_status,2132,2,STOPPED_AT,1307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
direction_id,2132.0,NaN,NaN,NaN,0.474203,0.499451,0.0,0.0,0.0,1.0,1.0
occupancy_status,2131,3,MANY_SEATS_AVAILABLE,1146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,2132.0,NaN,NaN,NaN,42.347651,0.015673,42.284885,42.33359,42.343057,42.362951,42.37346
longitude,2132.0,NaN,NaN,NaN,-71.090176,0.01416,-71.119396,-71.099646,-71.085452,-71.081028,-71.057001
bus_id,2132,24,y1759,226,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_id,2132.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
route_pattern_id,2132,2,1-_-0,1121,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Calidad de datos
Se detectan problemas antes de modificar nada: nulos, duplicados y columnas constantes

In [23]:
# Nulos
nulos = df.isna().sum()
nulos = nulos[nulos > 0].to_frame("nulos")
nulos["%"] = (nulos["nulos"] / len(df) * 100).round(2)
display(nulos)

# Duplicados
print("Filas duplicadas:", df.duplicated().sum())
print("Duplicados por (bus_id, date_time):", df.duplicated(subset=["bus_id", "date_time"]).sum())

# Columnas constantes
constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Columnas constantes:", constantes)

,nulos,%
bearing,479,22.47
occupancy_status,1,0.05
temperature_2m,3,0.14
precipitation,3,0.14
relative_humidity_2m,3,0.14
weather_code,3,0.14
is_day,3,0.14


Filas duplicadas: 0
Duplicados por (bus_id, date_time): 0
Columnas constantes: ['route_id', 'agency_id', 'line_id', 'timezone', 'road_closure']


## 5. Limpieza y tipos de dato
- Se eliminan las columnas constantes (no aportan información).
- `id` es un identificador de fila, no una variable predictiva: se descarta.
- Se convierten tipos: `is_day` y `es_desvio` a booleano, categóricas a `category`.

In [24]:
# Constantes + id (la lista "constantes" viene de la etapa anterior)
cols_a_eliminar = constantes + ["id"]
df = df.drop(columns=cols_a_eliminar)
print("Eliminadas:", cols_a_eliminar)

# Tipos
def a_bool(s):
    return s.map({True: True, False: False, "true": True, "false": False, "True": True, "False": False})

df["is_day"] = a_bool(df["is_day"])
df["es_desvio"] = a_bool(df["es_desvio"]).astype(bool)

for col in ["current_status", "occupancy_status", "bus_id", "route_pattern_id", "shape_id", "direction_id"]:
    df[col] = df[col].astype("category")

df.dtypes

Eliminadas: ['route_id', 'agency_id', 'line_id', 'timezone', 'road_closure', 'id']


bearing                   float64
current_status           category
direction_id             category
occupancy_status         category
latitude                  float64
longitude                 float64
bus_id                   category
route_pattern_id         category
shape_id                 category
distancia_en_traza        float64
distancia_fuera_traza     float64
es_desvio                    bool
date_time                     str
temperature_2m            float64
precipitation             float64
relative_humidity_2m      float64
weather_code              float64
is_day                     object
current_speed               int64
free_flow_speed             int64
current_travel_time         int64
freeflow_travel_time        int64
confidence                float64
dtype: object

## 6. Variables temporales
`date_time` viene en UTC. Se convierte a hora local de la ciudad (según la columna `timezone` original) y se derivan hora, día de la semana, etc. La hora se codifica también de forma cíclica (seno/coseno) para que 23:59 y 00:00 queden cerca.

In [25]:
tz_local = df_raw["timezone"].iloc[0]          # America/New_York
df["fecha_utc"] = pd.to_datetime(df["date_time"], utc=True)
df["fecha_local"] = df["fecha_utc"].dt.tz_convert(tz_local)
df = df.drop(columns=["date_time"])

df["hora"] = df["fecha_local"].dt.hour
df["minuto"] = df["fecha_local"].dt.minute
df["dia_semana"] = df["fecha_local"].dt.dayofweek   # 0 = lunes
df["es_finde"] = df["dia_semana"] >= 5
hora_dec = df["hora"] + df["minuto"] / 60
df["hora_sin"] = np.sin(2 * np.pi * hora_dec / 24)
df["hora_cos"] = np.cos(2 * np.pi * hora_dec / 24)

# Orden cronológico por colectivo (útil para trayectorias)
df = df.sort_values(["bus_id", "fecha_utc"]).reset_index(drop=True)
print("Rango:", df["fecha_local"].min(), "→", df["fecha_local"].max())
df[["fecha_local", "hora", "dia_semana", "es_finde", "hora_sin", "hora_cos"]].head()

Rango: 2026-11-04 10:08:21.320843-05:00 → 2026-11-04 17:24:08.329327-05:00


,fecha_local,hora,dia_semana,es_finde,hora_sin,hora_cos
0,2026-11-04 13:41:45.206932-05:00,13,2,False,-0.426569,-0.904455
1,2026-11-04 13:43:21.805754-05:00,13,2,False,-0.434445,-0.900698
2,2026-11-04 13:44:41.749806-05:00,13,2,False,-0.438371,-0.898794
3,2026-11-04 13:47:37.149023-05:00,13,2,False,-0.450098,-0.892979
4,2026-11-04 13:49:26.253906-05:00,13,2,False,-0.457874,-0.889017


## 7. Valores faltantes
Estrategia por columna:
- **Clima** (`temperature_2m`, `precipitation`, `relative_humidity_2m`, `weather_code`, `is_day`): son pocos nulos y la variable cambia lento en el tiempo, así que se rellenan con el valor más cercano en el tiempo.
- **`bearing`** (rumbo): hay muchos nulos (~22%). No se inventa el valor: se crea un indicador `bearing_faltante` y se deja el NaN.
- **`occupancy_status`**: se rellena con una categoría `UNKNOWN`.

In [26]:
# Clima: ordenar por tiempo y propagar
cols_clima = ["temperature_2m", "precipitation", "relative_humidity_2m", "weather_code", "is_day"]
df = df.sort_values("fecha_utc")
df[cols_clima] = df[cols_clima].ffill().bfill()
df["is_day"] = df["is_day"].astype(bool)

# Bearing: indicador de faltante
df["bearing_faltante"] = df["bearing"].isna()

# Ocupación
df["occupancy_status"] = (
    df["occupancy_status"].cat.add_categories("UNKNOWN").fillna("UNKNOWN")
)

df = df.sort_values(["bus_id", "fecha_utc"]).reset_index(drop=True)
print(df.isna().sum()[lambda s: s > 0])

bearing    479
dtype: int64


## 8. Desvíos de la traza
`es_desvio` marca las muestras en que el colectivo está lejos de su recorrido. En la muestra, las filas normales no superan ~65 de `distancia_fuera_traza` (unidad a confirmar, probablemente metros), mientras que los desvíos llegan a ~5000. En esas filas la posición proyectada sobre la traza deja de ser confiable para entrenar, por eso se descartan por defecto (`ELIMINAR_DESVIOS = True`).

In [27]:
n_desvios = df["es_desvio"].sum()
print("Filas con es_desvio:", n_desvios)
print("Máx. distancia_fuera_traza en filas normales:", df.loc[~df["es_desvio"], "distancia_fuera_traza"].max().round(1))

if ELIMINAR_DESVIOS:
    df = df[~df["es_desvio"]].reset_index(drop=True)
    print(f"Se descartaron {n_desvios} filas")

Filas con es_desvio: 85
Máx. distancia_fuera_traza en filas normales: 65.4
Se descartaron 85 filas


## 9. Ingeniería de variables
Variables derivadas que resumen mejor el estado del tránsito y el movimiento:
- `ratio_congestion`: tiempo de viaje actual / tiempo en flujo libre (>1 = más lento de lo normal).
- `ratio_velocidad`: velocidad actual / velocidad libre.
- `demora_s`: segundos de demora respecto al flujo libre.
- `id_tramo`: tramo continuo de un colectivo. Se corta si pasan más de `UMBRAL_CORTE_S` segundos entre muestras o si cambia el sentido.
- `dt_s`: segundos desde la muestra anterior, dentro del mismo tramo.
- `avance_km`: kilómetros avanzados sobre la traza desde la muestra anterior (`distancia_en_traza` está en km).
- `vel_real_kmh`: velocidad estimada del colectivo a partir del avance. Queda en NaN si el avance es negativo o supera `VEL_MAX_KMH` (ruido de la proyección).
- `ocupacion_ord`: ocupación codificada de forma ordinal.
- `detenido`: 1 si el colectivo está detenido en una parada.

In [ ]:
df["ratio_congestion"] = df["current_travel_time"] / df["freeflow_travel_time"]
df["ratio_velocidad"] = df["current_speed"] / df["free_flow_speed"]
df["demora_s"] = df["current_travel_time"] - df["freeflow_travel_time"]

# Tramos continuos por colectivo: empieza uno nuevo si es la primera muestra del bus,
# pasó más de UMBRAL_CORTE_S desde la anterior, o cambió el shape (cambio de sentido).
g = df.groupby("bus_id", observed=True)
dt_prev = g["fecha_utc"].diff().dt.total_seconds()
shape_prev = g["shape_id"].shift().astype("float")
df["nuevo_tramo"] = dt_prev.isna() | (dt_prev > UMBRAL_CORTE_S) | (df["shape_id"].astype("float") != shape_prev)
df["id_tramo"] = df["nuevo_tramo"].groupby(df["bus_id"], observed=True).cumsum()

# Movimiento dentro de cada tramo
gt = df.groupby(["bus_id", "id_tramo"], observed=True)
df["dt_s"] = gt["fecha_utc"].diff().dt.total_seconds()
df["avance_km"] = gt["distancia_en_traza"].diff()          # distancia_en_traza en km
df["vel_real_kmh"] = df["avance_km"] / (df["dt_s"] / 3600)
# Avance negativo o velocidad imposible = ruido de la proyección sobre la traza
df.loc[(df["avance_km"] < 0) | (df["vel_real_kmh"] > VEL_MAX_KMH), "vel_real_kmh"] = np.nan

# Ocupación ordinal (estándar GTFS-realtime)
orden_ocupacion = {
    "EMPTY": 0, "MANY_SEATS_AVAILABLE": 1, "FEW_SEATS_AVAILABLE": 2,
    "STANDING_ROOM_ONLY": 3, "CRUSHED_STANDING_ROOM_ONLY": 4, "FULL": 5,
    "UNKNOWN": np.nan,
}
df["ocupacion_ord"] = df["occupancy_status"].astype(str).map(orden_ocupacion)

# Indicador de parada
df["detenido"] = (df["current_status"] == "STOPPED_AT").astype(int)

df[["ratio_congestion", "ratio_velocidad", "demora_s", "dt_s", "avance_km",
    "vel_real_kmh", "ocupacion_ord", "detenido"]].describe().T

,count,mean,std,min,25%,50%,75%,max
ratio_congestion,2047.0,1.377169,0.435637,1.000000,1.000000,1.291262,1.500000,3.838235
ratio_velocidad,2047.0,0.777832,0.176160,0.263158,0.666667,0.772727,1.000000,1.000000
demora_s,2047.0,62.078163,61.133251,0.000000,0.000000,49.000000,96.000000,310.000000
dt_s,1939.0,101.659586,26.327964,65.065247,80.939111,92.566048,122.691773,192.940412
avance_km,1939.0,0.235168,0.318062,-1.271128,0.081393,0.197437,0.329780,4.113432
vel_real_kmh,1823.0,8.618088,6.871850,0.000000,3.876721,7.725175,12.133129,78.197742
ocupacion_ord,2047.0,1.877382,1.307214,1.000000,1.000000,1.000000,2.000000,5.000000
detenido,2047.0,0.597460,0.490529,0.000000,0.000000,1.000000,1.000000,1.000000


## 10. Exportación del dataset procesado
Se guarda el resultado en `data/processed/`. Esta carpeta **no se sube al repo** (agregar `data/` al `.gitignore`).

In [29]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_final = df.drop(columns=["fecha_utc"])   # fecha_local queda como referencia temporal
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado en {OUTPUT_PATH} | {df_final.shape[0]} filas × {df_final.shape[1]} columnas")

Guardado en ../data/processed/dataset_procesado.csv | 2047 filas × 40 columnas
